# 06_Product Analytics

## 1. Objectifs de l'analyse

Ce notebook étudie la contribution des produits à la performance commerciale et à la rentabilité estimée.

Les analyses couvrent :
- la performance par catégorie et sous-catégorie ;
- l'identification des produits déficitaires ;
- les prix de vente observés et les coûts standards ;
- l'impact des transactions réalisées sous le coût standard.

In [2]:
# Importer les bibliothèques nécessaires
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Définir le chemin du projet
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Charger les cinq tables préparées
dim_date = pd.read_csv(
    PROCESSED_DIR / "DimDate.csv",
    parse_dates=["Date"]
)

dim_customer = pd.read_csv(
    PROCESSED_DIR / "DimCustomer.csv",
    parse_dates=["BirthDate", "DateFirstPurchase"]
)

dim_product = pd.read_csv(
    PROCESSED_DIR / "DimProduct.csv",
    parse_dates=["StartDate"]
)

fact_sales = pd.read_csv(
    PROCESSED_DIR / "FactInternetSales.csv",
    parse_dates=["OrderDate", "ShipDate"]
)

dim_territory = pd.read_csv(
    PROCESSED_DIR / "DimSalesTerritory.csv"
)

print("Chargement des données préparées terminé.")

Chargement des données préparées terminé.


## 2. Enrichissement des ventes avec les informations produits

In [4]:
# Sélectionner les informations nécessaires à l'analyse
colonnes_produits = [
    "ProductKey",
    "ProductName",
    "SubCategory",
    "Category"
]

# Afficher les informations produits
produits_analyse = dim_product[colonnes_produits].copy()

display(produits_analyse.head())

,ProductKey,ProductName,SubCategory,Category
0,1,Adjustable Race,NaN,NaN
1,2,Bearing Ball,NaN,NaN
2,3,BB Ball Bearing,NaN,NaN
3,4,Headset Ball Bearings,NaN,NaN
4,5,Blade,NaN,NaN


In [5]:
# Ajouter les informations produits aux lignes de vente
fact_sales_enriched = fact_sales.merge(
    produits_analyse,
    on="ProductKey",
    how="left",
    validate="many_to_one"
)

# Vérifier le résultat de la jointure
print("Dimensions avant jointure :", fact_sales.shape)
print("Dimensions après jointure :", fact_sales_enriched.shape)

print(
    "Nombre de lignes sans correspondance produit :",
    fact_sales_enriched["ProductName"].isna().sum()
)

display(
    fact_sales_enriched[
        ["ProductKey", "ProductName", "SubCategory", "Category"]
    ].head()
)

Dimensions avant jointure : (58189, 15)
Dimensions après jointure : (58189, 18)
Nombre de lignes sans correspondance produit : 0


,ProductKey,ProductName,SubCategory,Category
0,310,"Road-150 Red, 62",Road Bikes,Bikes
1,346,"Mountain-100 Silver, 44",Mountain Bikes,Bikes
2,346,"Mountain-100 Silver, 44",Mountain Bikes,Bikes
3,336,"Road-650 Black, 62",Road Bikes,Bikes
4,346,"Mountain-100 Silver, 44",Mountain Bikes,Bikes


In [6]:
# Vérifier la conservation des lignes après jointure
print("Nombre de lignes avant jointure :", len(fact_sales))
print("Nombre de lignes après jointure :", len(fact_sales_enriched))

# Vérifier les correspondances produits
print(
    "\nNombre de lignes sans nom de produit :",
    fact_sales_enriched["ProductName"].isna().sum()
)

# Mesurer les catégories manquantes
print(
    "Nombre de lignes sans catégorie :",
    fact_sales_enriched["Category"].isna().sum()
)

print(
    "Nombre de lignes sans sous-catégorie :",
    fact_sales_enriched["SubCategory"].isna().sum()
)

# Calculer la part des ventes sans catégorie
part_sans_categorie = (
    fact_sales_enriched["Category"].isna().mean() * 100
)

print(
    f"\nPart des lignes de vente sans catégorie : {part_sans_categorie:.2f} %"
)

Nombre de lignes avant jointure : 58189
Nombre de lignes après jointure : 58189

Nombre de lignes sans nom de produit : 0
Nombre de lignes sans catégorie : 0
Nombre de lignes sans sous-catégorie : 0

Part des lignes de vente sans catégorie : 0.00 %


## 3. Performance par catégorie et sous-catégorie

In [8]:
# Agréger les indicateurs financiers par catégorie
rentabilite_categorie = (
    fact_sales_enriched
    .groupby("Category")
    .agg(
        Chiffre_Affaires=("SalesAmount", "sum"),
        Cout_Total_Estime=("CalculatedTotalCost", "sum"),
        Nombre_Commandes=("SalesOrderNumber", "nunique"),
        Nombre_Lignes=("SalesOrderNumber", "count"),
        Quantite_Vendue=("OrderQuantity", "sum"),
        Nombre_Produits=("ProductKey", "nunique")
    )
    .reset_index()
)

# Calculer le bénéfice brut estimé
rentabilite_categorie["Benefice_Brut_Estime"] = (
    rentabilite_categorie["Chiffre_Affaires"]
    - rentabilite_categorie["Cout_Total_Estime"]
)

# Calculer le taux de marque estimé
rentabilite_categorie["Taux_Marque_Estime_%"] = (
    rentabilite_categorie["Benefice_Brut_Estime"]
    / rentabilite_categorie["Chiffre_Affaires"] * 100
)

# Trier les catégories par chiffre d'affaires décroissant
rentabilite_categorie = rentabilite_categorie.sort_values(
    "Chiffre_Affaires",
    ascending=False
)

display(rentabilite_categorie.round(2))

,Category,Chiffre_Affaires,Cout_Total_Estime,Nombre_Commandes,Nombre_Lignes,Quantite_Vendue,Nombre_Produits,Benefice_Brut_Estime,Taux_Marque_Estime_%
1,Bikes,28318144.65,26374642.79,15205,15205,23872,116,1943501.86,6.86
0,Accessories,667015.32,393152.35,17341,34319,53889,22,273862.97,41.06
2,Clothing,322676.62,305685.47,7110,8665,13560,20,16991.15,5.27


In [9]:
# Agréger les indicateurs financiers par sous-catégorie
rentabilite_sous_categorie = (
    fact_sales_enriched
    .groupby(["Category", "SubCategory"])
    .agg(
        Chiffre_Affaires=("SalesAmount", "sum"),
        Cout_Total_Estime=("CalculatedTotalCost", "sum"),
        Nombre_Commandes=("SalesOrderNumber", "nunique"),
        Nombre_Lignes=("SalesOrderNumber", "count"),
        Quantite_Vendue=("OrderQuantity", "sum"),
        Nombre_Produits=("ProductKey", "nunique")
    )
    .reset_index()
)

# Calculer le bénéfice brut estimé
rentabilite_sous_categorie["Benefice_Brut_Estime"] = (
    rentabilite_sous_categorie["Chiffre_Affaires"]
    - rentabilite_sous_categorie["Cout_Total_Estime"]
)

# Calculer le taux de marque estimé
rentabilite_sous_categorie["Taux_Marque_Estime_%"] = (
    rentabilite_sous_categorie["Benefice_Brut_Estime"]
    / rentabilite_sous_categorie["Chiffre_Affaires"] * 100
)

# Trier par chiffre d'affaires décroissant
rentabilite_sous_categorie = rentabilite_sous_categorie.sort_values(
    "Chiffre_Affaires",
    ascending=False
)

display(rentabilite_sous_categorie.round(2))

,Category,SubCategory,Chiffre_Affaires,Cout_Total_Estime,Nombre_Commandes,Nombre_Lignes,Quantite_Vendue,Nombre_Produits,Benefice_Brut_Estime,Taux_Marque_Estime_%
9,Bikes,Road Bikes,14520584.04,14098238.88,8068,8068,12695,60,422345.16,2.91
8,Bikes,Mountain Bikes,9952759.56,8517720.75,4970,4970,7767,34,1435038.81,14.42
10,Bikes,Touring Bikes,3844801.05,3758683.16,2167,2167,3410,22,86117.89,2.24
7,Accessories,Tires and Tubes,231299.76,134890.84,9272,16299,25518,11,96408.92,41.68
5,Accessories,Helmets,215923.29,126740.82,6171,6171,9685,3,89182.47,41.30
13,Clothing,Jerseys,165066.21,202581.86,3179,3179,5068,8,-37515.65,-22.73
14,Clothing,Shorts,67050.42,39028.86,958,958,1491,3,28021.56,41.79
2,Accessories,Bottles and Cages,55030.81,32116.40,4583,7719,12055,3,22914.41,41.64
4,Accessories,Fenders,44267.72,26626.20,2014,2014,3239,1,17641.52,39.85
6,Accessories,Hydration Packs,38822.94,23116.52,706,706,1124,1,15706.42,40.46


## 4. Identification des produits déficitaires

In [11]:
# Identifier les produits appartenant aux deux sous-catégories déficitaires
produits_deficitaires = fact_sales_enriched[
    fact_sales_enriched["SubCategory"].isin(["Jerseys", "Caps"])
]

# Calculer les indicateurs financiers par produit
rentabilite_produits_deficitaires = (
    produits_deficitaires
    .groupby(["Category", "SubCategory", "ProductKey", "ProductName"])
    .agg(
        Chiffre_Affaires=("SalesAmount", "sum"),
        Cout_Total_Estime=("CalculatedTotalCost", "sum"),
        Nombre_Commandes=("SalesOrderNumber", "nunique"),
        Quantite_Vendue=("OrderQuantity", "sum")
    )
    .reset_index()
)

# Calculer le bénéfice brut estimé
rentabilite_produits_deficitaires["Benefice_Brut_Estime"] = (
    rentabilite_produits_deficitaires["Chiffre_Affaires"]
    - rentabilite_produits_deficitaires["Cout_Total_Estime"]
)

# Calculer le taux de marque estimé
rentabilite_produits_deficitaires["Taux_Marque_Estime_%"] = (
    rentabilite_produits_deficitaires["Benefice_Brut_Estime"]
    / rentabilite_produits_deficitaires["Chiffre_Affaires"] * 100
)

# Trier les produits du plus déficitaire au plus rentable
rentabilite_produits_deficitaires = (
    rentabilite_produits_deficitaires
    .sort_values("Benefice_Brut_Estime", ascending=True)
)

display(rentabilite_produits_deficitaires.round(2))

,Category,SubCategory,ProductKey,ProductName,Chiffre_Affaires,Cout_Total_Estime,Nombre_Commandes,Quantite_Vendue,Benefice_Brut_Estime,Taux_Marque_Estime_%
5,Clothing,Jerseys,488,"Short-Sleeve Classic Jersey, S",21326.05,26647.84,395,641,-5321.79,-24.95
2,Clothing,Jerseys,231,"Long-Sleeve Logo Jersey, M",20895.82,26097.78,418,678,-5201.96,-24.89
1,Clothing,Jerseys,228,"Long-Sleeve Logo Jersey, S",20045.99,25058.49,401,651,-5012.50,-25.00
3,Clothing,Jerseys,234,"Long-Sleeve Logo Jersey, L",21595.68,26174.76,432,680,-4579.08,-21.20
8,Clothing,Jerseys,491,"Short-Sleeve Classic Jersey, XL",21056.10,25566.96,390,615,-4510.86,-21.42
4,Clothing,Jerseys,237,"Long-Sleeve Logo Jersey, XL",19546.09,23980.70,391,623,-4434.61,-22.69
6,Clothing,Jerseys,489,"Short-Sleeve Classic Jersey, M",20840.14,25109.67,386,604,-4269.53,-20.49
7,Clothing,Jerseys,490,"Short-Sleeve Classic Jersey, L",19760.34,23945.64,366,576,-4185.30,-21.18
0,Clothing,Caps,225,AWC Logo Cap,18834.05,21999.07,2095,3178,-3165.02,-16.80


## 5. Analyse des prix et des coûts des produits déficitaires

In [13]:
# Examiner les colonnes disponibles dans les ventes enrichies
print("Colonnes de fact_sales_enriched :")

for colonne in fact_sales_enriched.columns:
    print("-", colonne)

Colonnes de fact_sales_enriched :
- ProductKey
- OrderDate
- ShipDate
- CustomerKey
- PromotionKey
- SalesTerritoryKey
- SalesOrderNumber
- SalesOrderLineNumber
- OrderQuantity
- UnitPrice
- TotalProductCost
- SalesAmount
- TaxAmt
- CalculatedTotalCost
- StandardCost
- ProductName
- SubCategory
- Category


In [14]:
# Rechercher les colonnes dont le nom évoque une remise
colonnes_remise = [
    colonne for colonne in fact_sales_enriched.columns
    if "discount" in colonne.lower()
    or "remise" in colonne.lower()
]

print("Colonnes liées aux remises :", colonnes_remise)

Colonnes liées aux remises : []


In [15]:
# Calculer les prix et coûts unitaires moyens pondérés
analyse_prix_ponderes = (
    produits_deficitaires
    .groupby(["SubCategory", "ProductKey", "ProductName"])
    .agg(
        Chiffre_Affaires=("SalesAmount", "sum"),
        Cout_Total_Estime=("CalculatedTotalCost", "sum"),
        Quantite_Vendue=("OrderQuantity", "sum"),
        Cout_Standard_Unitaire=("StandardCost", "first")
    )
    .reset_index()
)

# Calculer le prix de vente moyen pondéré
analyse_prix_ponderes["Prix_Vente_Moyen_Pondere"] = (
    analyse_prix_ponderes["Chiffre_Affaires"]
    / analyse_prix_ponderes["Quantite_Vendue"]
)

# Calculer le coût unitaire estimé pondéré
analyse_prix_ponderes["Cout_Unitaire_Estime"] = (
    analyse_prix_ponderes["Cout_Total_Estime"]
    / analyse_prix_ponderes["Quantite_Vendue"]
)

# Calculer l'écart entre le prix et le coût
analyse_prix_ponderes["Ecart_Prix_Cout"] = (
    analyse_prix_ponderes["Prix_Vente_Moyen_Pondere"]
    - analyse_prix_ponderes["Cout_Unitaire_Estime"]
)

# Calculer le taux de marque estimé
analyse_prix_ponderes["Taux_Marque_Estime_%"] = (
    analyse_prix_ponderes["Ecart_Prix_Cout"]
    / analyse_prix_ponderes["Prix_Vente_Moyen_Pondere"] * 100
)

display(analyse_prix_ponderes.round(2))

,SubCategory,ProductKey,ProductName,Chiffre_Affaires,Cout_Total_Estime,Quantite_Vendue,Cout_Standard_Unitaire,Prix_Vente_Moyen_Pondere,Cout_Unitaire_Estime,Ecart_Prix_Cout,Taux_Marque_Estime_%
0,Caps,225,AWC Logo Cap,18834.05,21999.07,3178,6.92,5.93,6.92,-1.00,-16.80
1,Jerseys,228,"Long-Sleeve Logo Jersey, S",20045.99,25058.49,651,38.49,30.79,38.49,-7.70,-25.00
2,Jerseys,231,"Long-Sleeve Logo Jersey, M",20895.82,26097.78,678,38.49,30.82,38.49,-7.67,-24.89
3,Jerseys,234,"Long-Sleeve Logo Jersey, L",21595.68,26174.76,680,38.49,31.76,38.49,-6.73,-21.20
4,Jerseys,237,"Long-Sleeve Logo Jersey, XL",19546.09,23980.70,623,38.49,31.37,38.49,-7.12,-22.69
5,Jerseys,488,"Short-Sleeve Classic Jersey, S",21326.05,26647.84,641,41.57,33.27,41.57,-8.30,-24.95
6,Jerseys,489,"Short-Sleeve Classic Jersey, M",20840.14,25109.67,604,41.57,34.50,41.57,-7.07,-20.49
7,Jerseys,490,"Short-Sleeve Classic Jersey, L",19760.34,23945.64,576,41.57,34.31,41.57,-7.27,-21.18
8,Jerseys,491,"Short-Sleeve Classic Jersey, XL",21056.10,25566.96,615,41.57,34.24,41.57,-7.33,-21.42


In [16]:
# Examiner la distribution des prix de vente par produit
distribution_prix = (
    produits_deficitaires
    .groupby(["SubCategory", "ProductKey", "ProductName"])
    .agg(
        Prix_Min=("UnitPrice", "min"),
        Prix_Moyen=("UnitPrice", "mean"),
        Prix_Median=("UnitPrice", "median"),
        Prix_Max=("UnitPrice", "max"),
        Nombre_Prix_Differents=("UnitPrice", "nunique"),
        Nombre_Lignes=("UnitPrice", "count")
    )
    .reset_index()
)

display(distribution_prix.round(2))

,SubCategory,ProductKey,ProductName,Prix_Min,Prix_Moyen,Prix_Median,Prix_Max,Nombre_Prix_Differents,Nombre_Lignes
0,Caps,225,AWC Logo Cap,2.25,7.50,8.99,8.99,3,2095
1,Jerseys,228,"Long-Sleeve Logo Jersey, S",12.50,40.11,49.99,49.99,3,401
2,Jerseys,231,"Long-Sleeve Logo Jersey, M",12.50,40.09,49.99,49.99,3,418
3,Jerseys,234,"Long-Sleeve Logo Jersey, L",12.50,40.50,49.99,49.99,3,432
4,Jerseys,237,"Long-Sleeve Logo Jersey, XL",12.50,40.62,49.99,49.99,3,391
5,Jerseys,488,"Short-Sleeve Classic Jersey, S",13.50,43.64,53.99,53.99,3,395
6,Jerseys,489,"Short-Sleeve Classic Jersey, M",13.50,44.51,53.99,53.99,3,386
7,Jerseys,490,"Short-Sleeve Classic Jersey, L",13.50,43.70,53.99,53.99,3,366
8,Jerseys,491,"Short-Sleeve Classic Jersey, XL",13.50,44.23,53.99,53.99,3,390


In [17]:
# Identifier les ventes réalisées sous le coût standard
produits_deficitaires = produits_deficitaires.copy()

produits_deficitaires["Vente_Sous_Cout_Standard"] = (
    produits_deficitaires["UnitPrice"]
    < produits_deficitaires["StandardCost"]
)

# Comparer les ventes sous et au-dessus du coût standard
impact_prix = (
    produits_deficitaires
    .groupby("Vente_Sous_Cout_Standard")
    .agg(
        Nombre_Lignes=("SalesOrderNumber", "count"),
        Quantite_Vendue=("OrderQuantity", "sum"),
        Chiffre_Affaires=("SalesAmount", "sum"),
        Cout_Total_Estime=("CalculatedTotalCost", "sum")
    )
    .reset_index()
)

# Calculer le bénéfice brut estimé
impact_prix["Benefice_Brut_Estime"] = (
    impact_prix["Chiffre_Affaires"]
    - impact_prix["Cout_Total_Estime"]
)

# Calculer le taux de marque estimé
impact_prix["Taux_Marque_Estime_%"] = (
    impact_prix["Benefice_Brut_Estime"]
    / impact_prix["Chiffre_Affaires"] * 100
)

display(impact_prix.round(2))

,Vente_Sous_Cout_Standard,Nombre_Lignes,Quantite_Vendue,Chiffre_Affaires,Cout_Total_Estime,Benefice_Brut_Estime,Taux_Marque_Estime_%
0,False,3738,3738,128370.62,98845.38,29525.24,23.00
1,True,1536,4508,55529.64,125735.55,-70205.91,-126.43


In [18]:
# Analyser les pertes par produit pour les ventes sous le coût standard
pertes_par_produit = (
    produits_deficitaires[
        produits_deficitaires["Vente_Sous_Cout_Standard"]
    ]
    .groupby(["SubCategory", "ProductKey", "ProductName"])
    .agg(
        Nombre_Lignes=("SalesOrderNumber", "count"),
        Quantite_Vendue=("OrderQuantity", "sum"),
        Chiffre_Affaires=("SalesAmount", "sum"),
        Cout_Total_Estime=("CalculatedTotalCost", "sum")
    )
    .reset_index()
)

# Calculer le déficit estimé
pertes_par_produit["Perte_Estimee"] = (
    pertes_par_produit["Chiffre_Affaires"]
    - pertes_par_produit["Cout_Total_Estime"]
)

# Calculer la contribution de chaque produit aux pertes totales
perte_totale = abs(pertes_par_produit["Perte_Estimee"].sum())

pertes_par_produit["Contribution_Pertes_%"] = (
    abs(pertes_par_produit["Perte_Estimee"])
    / perte_totale * 100
)

# Classer les produits du plus gros déficit au plus faible
pertes_par_produit = pertes_par_produit.sort_values(
    "Perte_Estimee"
)

display(pertes_par_produit.round(2))

,SubCategory,ProductKey,ProductName,Nombre_Lignes,Quantite_Vendue,Chiffre_Affaires,Cout_Total_Estime,Perte_Estimee,Contribution_Pertes_%
5,Jerseys,488,"Short-Sleeve Classic Jersey, S",120,366,6478.80,15215.46,-8736.66,12.44
2,Jerseys,231,"Long-Sleeve Logo Jersey, M",134,394,6698.66,15165.97,-8467.31,12.06
1,Jerseys,228,"Long-Sleeve Logo Jersey, S",128,378,6398.72,14550.09,-8151.37,11.61
3,Jerseys,234,"Long-Sleeve Logo Jersey, L",136,384,6798.64,14781.04,-7982.40,11.37
8,Jerseys,491,"Short-Sleeve Classic Jersey, XL",113,338,6100.87,14051.44,-7950.57,11.32
6,Jerseys,489,"Short-Sleeve Classic Jersey, M",108,326,5830.92,13552.57,-7721.65,11.00
4,Jerseys,237,"Long-Sleeve Logo Jersey, XL",118,350,5898.82,13472.30,-7573.48,10.79
7,Jerseys,490,"Short-Sleeve Classic Jersey, L",116,326,6262.84,13552.57,-7289.73,10.38
0,Caps,225,AWC Logo Cap,563,1646,5061.37,11394.11,-6332.74,9.02


### Synthèse de l'analyse de la rentabilité des produits

L'analyse révèle une forte concentration du chiffre d'affaires sur la catégorie Bikes, qui représente environ 96,6 % des ventes. Cependant, les meilleures performances relatives ne sont pas systématiquement associées aux produits générant le plus de chiffre d'affaires. La catégorie Accessories affiche notamment un taux de marque estimé de 41,06 %, contre 6,86 % pour Bikes.

L'analyse par sous-catégorie met en évidence des différences importantes. Les Mountain Bikes présentent une rentabilité estimée supérieure à celle des Road Bikes, malgré un chiffre d'affaires moins élevé. À l'inverse, les Jerseys et les Caps enregistrent des résultats négatifs.

L'étude détaillée de ces produits montre que leurs prix de vente moyens pondérés sont inférieurs à leurs coûts standards unitaires estimés. Les huit références de Jerseys concentrent environ 92,22 % du déficit net estimé des neuf produits déficitaires.

L'analyse des transactions confirme également que les ventes réalisées sous le coût standard génèrent des pertes importantes, partiellement compensées par les ventes réalisées à des prix plus élevés.

Ces résultats suggèrent de surveiller particulièrement la politique de prix et les conditions commerciales appliquées aux Jerseys et aux Caps. Une analyse complémentaire des promotions et des remises serait nécessaire pour identifier les causes précises de ces écarts.

**Limite méthodologique :** les coûts utilisés sont des estimations calculées à partir du coût standard unitaire. Les résultats ne doivent donc pas être interprétés comme des bénéfices ou des pertes comptables réels.

## 6. Limites méthodologiques

- Les coûts totaux sont estimés à partir du coût standard unitaire multiplié par la quantité vendue.
- La table de ventes préparée ne contient pas de colonne de remise ou de promotion permettant d'attribuer les prix bas à une opération commerciale précise.
- Les prix minimums observés ne suffisent donc pas à démontrer l'existence d'une promotion.
- Les résultats de rentabilité doivent être interprétés comme des estimations et non comme des résultats comptables réels.